# 01 — EDA TrashCan 1.0 Instance (S2-DATA)

Thống kê 22 lớp trên annotation gốc (train + val gốc), kích thước bbox, mức mất cân bằng; nếu đã chạy
notebook 02 thì thêm số crop theo tập mới và ảnh crop mẫu. Hàm dùng: `src/preprocessing.build_object_table`,
`src/eda.py`. Phụ trách: Hùng (dự phòng: Yến).

> Notebook này **chỉ gọi hàm trong `src/`**: không `%%writefile`, không chép code nguồn vào cell, không token,
> không `git push`. Muốn sửa logic → sửa `src/` + `tests/` trên nhánh riêng và mở PR.
> Lưu notebook bằng *File → Save a copy in GitHub* vào **nhánh của mình**, xoá output trước khi commit.

## 1. Cài đặt

In [ ]:
# Cài đặt chung: Colab → mount Drive + clone repo; máy cá nhân → dùng repo đang mở.
import os, sys, subprocess, json, time
from pathlib import Path

BRANCH = "main"   # đổi thành "feat/s2-pipeline-end-to-end" nếu PR #21 chưa merge
REPO_URL = "https://github.com/phanyen06ym-ien/ocean-plastic-ml-starter.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/ocean-plastic-ml")   # thư mục chung của nhóm
    WORK = Path("/content")                                   # ổ đĩa Colab, đọc nhanh hơn Drive
    REPO = WORK / "repo"
    if not REPO.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE = REPO                                              # data/, runs/ đã nằm trong .gitignore
    WORK = REPO / "data"
os.chdir(REPO); sys.path.insert(0, str(REPO))
for sub in ["data/raw", "data/processed", "runs"]:
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)
print("Repo:", REPO, "| commit:",
      subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Dataset TrashCan gốc. Colab: giải nén .zip trên Drive ra ổ đĩa Colab. Máy cá nhân: giải nén sẵn vào data/raw/trashcan.
if IN_COLAB:
    RAW_ROOT = WORK / "trashcan"
    if not RAW_ROOT.exists():
        zips = sorted((DRIVE / "data/raw").glob("*.zip"))
        assert zips, f"Chưa có file .zip trong {DRIVE / 'data/raw'} — xem docs/colab_setup.md"
        RAW_ROOT.mkdir()
        subprocess.run(["unzip", "-q", str(zips[0]), "-d", str(RAW_ROOT)], check=True)
else:
    RAW_ROOT = WORK / "raw/trashcan"
assert RAW_ROOT.exists(), f"Không thấy dataset tại {RAW_ROOT}"
print("\n".join(str(p) for p in sorted(RAW_ROOT.rglob("instances_*_trashcan.json"))))

## 2. Bảng annotation và class mapping 22 lớp

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.common import load_contract
from src.preprocessing import build_object_table
from src import eda

contract = load_contract()
objects, mapping = build_object_table(RAW_ROOT, contract)
NAMES = [m["class_name"] for m in mapping]
PLASTIC = [m["class_name"] for m in mapping if m["plastic_related"]]
FIG, MET = REPO / "results/figures", REPO / "results/metrics"
FIG.mkdir(parents=True, exist_ok=True); MET.mkdir(parents=True, exist_ok=True)

print("Annotation:", len(objects), "| ảnh nguồn:", objects["image_id"].nunique(),
      "| video:", objects["video_id"].nunique(), "(Dataset Audit ghi nhận 312) | lớp:", len(mapping))
print("Annotation không tìm thấy file ảnh:", int((objects["image_path"] == "").sum()))
display(pd.DataFrame(mapping))

## 3. Phân bố lớp và mất cân bằng

In [ ]:
dist = eda.class_distribution(objects, by="source_split")
dist.to_csv(MET / "dataset_class_distribution.csv")
display(dist)
print("Tỉ lệ mất cân bằng (lớn nhất / nhỏ nhất):", round(eda.imbalance_ratio(objects), 1))
fig, _ = eda.plot_class_distribution(dist, highlight=PLASTIC, save_path=FIG / "eda_class_distribution.png")
display(fig); plt.close(fig)

## 4. Kích thước bbox (cạnh ngắn, pixel)

In [ ]:
display(eda.bbox_stats(objects))

## 5. Sau tiền xử lý (cần chạy notebook 02 trước — bỏ qua nếu chưa có)

In [ ]:
DATA_VERSION = contract["dataset"]["data_version"]
PROCESSED = WORK / "processed" / DATA_VERSION
PROCESSED_ZIP = DRIVE / "data/processed" / f"{DATA_VERSION}.zip"
if IN_COLAB and not PROCESSED.exists() and PROCESSED_ZIP.exists():
    PROCESSED.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-q", str(PROCESSED_ZIP), "-d", str(PROCESSED.parent)], check=True)

if (PROCESSED / "split_manifest.csv").exists():
    manifest = pd.read_csv(PROCESSED / "split_manifest.csv")
    split_counts = pd.read_csv(PROCESSED / "class_split_counts.csv", index_col=0)
    split_counts.to_csv(MET / "dataset_split_counts.csv")
    display(split_counts)
    fig, _ = eda.plot_split_counts(split_counts, save_path=FIG / "eda_split_counts.png")
    display(fig); plt.close(fig)
    fig, _ = eda.plot_sample_crops(manifest, PROCESSED, NAMES, per_class=4, save_path=FIG / "eda_sample_crops.png")
    display(fig); plt.close(fig)
else:
    print("Chưa có", PROCESSED, "— chạy notebook 02 rồi quay lại mục này.")